##Data Prep and Augmentation (Originally ran via Google Colab)

SSD Fruit Detection

Data Used

LVIS Fruits And Vegetables Dataset
https://www.kaggle.com/datasets/henningheyen/lvis-fruits-and-vegetables-dataset

Fruit Detection Dataset
https://www.kaggle.com/datasets/lakshaytyagi01/fruit-detection

Fruits Images Dataset: Object Detection
https://www.kaggle.com/datasets/afsananadia/fruits-images-dataset-object-detection

Fruit and Vegetables
https://www.kaggle.com/datasets/youssefsalahzakria/fruit-and-vegetables-classification

In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
from xml.dom import minidom
from PIL import Image


FRUIT_CLASSES = {
    1: "apple",
    2: "apricot",
    5: "avocado",
    6: "banana",
    9: "blackberry",
    10: "blueberry",
    13: "cantaloupe",
    18: "cherry",
    21: "clementine",
    22: "coconut",
    25: "date",
    27: "fig",
    30: "strawberry",
    32: "grape",
    35: "tomato",
    36: "kiwi fruit",
    37: "lemon",
    39: "lime",
    40: "mandarin orange",
    41: "melon",
    44: "orange",
    45: "papaya",
    47: "peach",
    48: "pear",
    49: "persimmon",
    51: "pineapple",
    53: "prune",
    56: "raspberry",
    57: "strawberry",
    59: "tomato",
    61: "watermelon"
}

# Remap class IDs
REMAPPED_CLASSES = {orig_id: i for i, orig_id in enumerate(FRUIT_CLASSES)}
REVERSE_CLASS_MAP = {i: FRUIT_CLASSES[orig_id] for orig_id, i in REMAPPED_CLASSES.items()}

# Output
OUTPUT_IMAGE_DIR = '/content/lvis_pascal/images'
OUTPUT_ANNOT_DIR = '/content/lvis_pascal/annotations'

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_ANNOT_DIR, exist_ok=True)

def create_voc_xml(img_path, img_width, img_height, objects, output_path):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'filename').text = os.path.basename(img_path)
    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(img_width)
    ET.SubElement(size, 'height').text = str(img_height)
    ET.SubElement(size, 'depth').text = '3'

    for obj in objects:
        obj_elem = ET.SubElement(annotation, 'object')
        ET.SubElement(obj_elem, 'name').text = obj['name']
        bbox = ET.SubElement(obj_elem, 'bndbox')
        ET.SubElement(bbox, 'xmin').text = str(obj['xmin'])
        ET.SubElement(bbox, 'ymin').text = str(obj['ymin'])
        ET.SubElement(bbox, 'xmax').text = str(obj['xmax'])
        ET.SubElement(bbox, 'ymax').text = str(obj['ymax'])

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent='  ')
    with open(output_path, 'w') as f:
        f.write(xml_str)

def convert_yolo_to_voc(yolo_txt_path, image_path):
    with open(yolo_txt_path, 'r') as f:
        lines = f.readlines()

    img = Image.open(image_path)
    width, height = img.size
    objects = []

    for line in lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        if cls_id not in REMAPPED_CLASSES:
            continue

        x_center, y_center, w, h = map(float, parts[1:])
        x_center *= width
        y_center *= height
        w *= width
        h *= height

        xmin = int(x_center - w / 2)
        ymin = int(y_center - h / 2)
        xmax = int(x_center + w / 2)
        ymax = int(y_center + h / 2)

        objects.append({
            'name': REVERSE_CLASS_MAP[REMAPPED_CLASSES[cls_id]],
            'xmin': max(0, xmin),
            'ymin': max(0, ymin),
            'xmax': min(width, xmax),
            'ymax': min(height, ymax)
        })

    return objects, width, height

#  train/val/test
ROOT_DIR = "/content/LVIS_Fruits_And_Vegetables"
for split in ['train/train', 'val/val', 'test']:
    image_dir = os.path.join(ROOT_DIR, 'images', split)
    label_dir = os.path.join(ROOT_DIR, 'labels', split)
    image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))

    for img_path in image_paths:
        file_id = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(label_dir, file_id + '.txt')
        if not os.path.exists(label_path):
            continue

        try:
            objects, w, h = convert_yolo_to_voc(label_path, img_path)
            if len(objects) == 0:
                continue

            # Copy image
            out_img_path = os.path.join(OUTPUT_IMAGE_DIR, f"{file_id}.jpg")
            shutil.copyfile(img_path, out_img_path)

            # Write Pascal XML
            out_xml_path = os.path.join(OUTPUT_ANNOT_DIR, f"{file_id}.xml")
            create_voc_xml(out_img_path, w, h, objects, out_xml_path)
        except Exception as e:
            print(f"[ERROR] {img_path} — {e}")


In [ ]:
label_map_path = "/content/lvis_pascal/label_map.pbtxt"
with open(label_map_path, "w") as f:
    for idx, name in REVERSE_CLASS_MAP.items():
        f.write("item {\n")
        f.write(f"  id: {idx + 1}\n")
        f.write(f"  name: '{name}'\n")
        f.write("}\n")


In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
from xml.dom import minidom
from PIL import Image

CLASS_NAMES = ['Apple', 'Banana', 'Grape', 'Orange', 'Pineapple', 'Watermelon']
REMAPPED_CLASSES = {i: i for i in range(len(CLASS_NAMES))}

# Reverse map... label_map.pbtxt
REVERSE_CLASS_MAP = {v: k for k, v in REMAPPED_CLASSES.items()}
REVERSE_CLASS_NAMES = {i: name.lower() for i, name in enumerate(CLASS_NAMES)}

# Output
OUTPUT_IMAGE_DIR = '/content/fruit_detection_pascal/images'
OUTPUT_ANNOT_DIR = '/content/fruit_detection_pascal/annotations'
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_ANNOT_DIR, exist_ok=True)

def create_voc_xml(img_path, img_width, img_height, objects, output_path):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'filename').text = os.path.basename(img_path)
    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(img_width)
    ET.SubElement(size, 'height').text = str(img_height)
    ET.SubElement(size, 'depth').text = '3'

    for obj in objects:
        obj_elem = ET.SubElement(annotation, 'object')
        ET.SubElement(obj_elem, 'name').text = obj['name']
        bbox = ET.SubElement(obj_elem, 'bndbox')
        ET.SubElement(bbox, 'xmin').text = str(obj['xmin'])
        ET.SubElement(bbox, 'ymin').text = str(obj['ymin'])
        ET.SubElement(bbox, 'xmax').text = str(obj['xmax'])
        ET.SubElement(bbox, 'ymax').text = str(obj['ymax'])

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent='  ')
    with open(output_path, 'w') as f:
        f.write(xml_str)

def convert_yolo_to_voc(yolo_txt_path, image_path):
    with open(yolo_txt_path, 'r') as f:
        lines = f.readlines()

    img = Image.open(image_path)
    width, height = img.size
    objects = []

    for line in lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        if cls_id not in REMAPPED_CLASSES:
            continue

        x_center, y_center, w, h = map(float, parts[1:])
        x_center *= width
        y_center *= height
        w *= width
        h *= height

        xmin = int(x_center - w / 2)
        ymin = int(y_center - h / 2)
        xmax = int(x_center + w / 2)
        ymax = int(y_center + h / 2)

        objects.append({
            'name': REVERSE_CLASS_NAMES[cls_id],
            'xmin': max(0, xmin),
            'ymin': max(0, ymin),
            'xmax': min(width, xmax),
            'ymax': min(height, ymax)
        })

    return objects, width, height

# Loop through train/valid/test
ROOT = '/content/Fruit_Detection'
splits = ['train', 'valid', 'test']

for split in splits:
    image_dir = os.path.join(ROOT, split, 'images')
    label_dir = os.path.join(ROOT, split, 'labels')
    image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))

    for img_path in image_paths:
        file_id = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(label_dir, file_id + '.txt')
        if not os.path.exists(label_path):
            continue

        try:
            objects, w, h = convert_yolo_to_voc(label_path, img_path)
            if len(objects) == 0:
                continue

            # Copy image
            out_img_path = os.path.join(OUTPUT_IMAGE_DIR, f"{file_id}.jpg")
            shutil.copyfile(img_path, out_img_path)

            # Write Pascal XML
            out_xml_path = os.path.join(OUTPUT_ANNOT_DIR, f"{file_id}.xml")
            create_voc_xml(out_img_path, w, h, objects, out_xml_path)
        except Exception as e:
            print(f"[ERROR] {img_path} — {e}")


In [ ]:
label_map_path = "/content/fruit_detection_pascal/label_map.pbtxt"
with open(label_map_path, "w") as f:
    for idx, name in REVERSE_CLASS_NAMES.items():
        f.write("item {\n")
        f.write(f"  id: {idx + 1}\n")
        f.write(f"  name: '{name}'\n")
        f.write("}\n")


In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
from xml.dom import minidom
from PIL import Image
import re

# Configure paths
ROOT_DIR = "/content/Fruit_Images_Dataset"
TRAIN_DIR = os.path.join(ROOT_DIR, "Train File", "Train File")
TEST_DIR = os.path.join(ROOT_DIR, "Test File", "Test File")

OUTPUT_IMG_DIR = "/content/fruit_images_pascal/images"
OUTPUT_ANN_DIR = "/content/fruit_images_pascal/annotations"
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_ANN_DIR, exist_ok=True)

# Class ID mapping
CLASS_NAMES = [
    'apple', 'banana', 'grapes', 'guava', 'hogplum', 'jackfruit',
    'litchi', 'mango', 'orange', 'papaya'
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# Reverse lookup for label map
REVERSE_CLASS_MAP = {v: k for k, v in CLASS_TO_ID.items()}

# Helper: Pascal VOC writer
def create_voc_xml(img_path, img_width, img_height, objects, output_path):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'filename').text = os.path.basename(img_path)
    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(img_width)
    ET.SubElement(size, 'height').text = str(img_height)
    ET.SubElement(size, 'depth').text = '3'

    for obj in objects:
        obj_elem = ET.SubElement(annotation, 'object')
        ET.SubElement(obj_elem, 'name').text = obj['name']
        bbox = ET.SubElement(obj_elem, 'bndbox')
        ET.SubElement(bbox, 'xmin').text = str(obj['xmin'])
        ET.SubElement(bbox, 'ymin').text = str(obj['ymin'])
        ET.SubElement(bbox, 'xmax').text = str(obj['xmax'])
        ET.SubElement(bbox, 'ymax').text = str(obj['ymax'])

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent='  ')
    with open(output_path, 'w') as f:
        f.write(xml_str)

#  YOLO to VOC conversion
def convert_yolo_to_voc(yolo_txt_path, image_path, label_name):
    with open(yolo_txt_path, 'r') as f:
        lines = f.readlines()

    img = Image.open(image_path)
    width, height = img.size
    objects = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        cls_id = int(parts[0])  #
        x_center, y_center, w, h = map(float, parts[1:])
        x_center *= width
        y_center *= height
        w *= width
        h *= height

        xmin = int(x_center - w / 2)
        ymin = int(y_center - h / 2)
        xmax = int(x_center + w / 2)
        ymax = int(y_center + h / 2)

        objects.append({
            'name': label_name,
            'xmin': max(0, xmin),
            'ymin': max(0, ymin),
            'xmax': min(width, xmax),
            'ymax': min(height, ymax)
        })

    return objects, width, height

# Main conversion loop
def process_folder(folder_path):
    jpg_files = glob.glob(os.path.join(folder_path, "*.jpg"))
    for img_path in jpg_files:
        file_name = os.path.basename(img_path)
        base_name = os.path.splitext(file_name)[0]

        # Extract class name from filename
        match = re.search(r'_(\w+)$', base_name)
        if not match:
            continue
        label_raw = match.group(1).lower()
        label = label_raw.strip().lower()

        if label not in CLASS_TO_ID:
            continue

        txt_path = os.path.join(folder_path, f"{base_name}.txt")
        if not os.path.exists(txt_path):
            continue

        try:
            objects, w, h = convert_yolo_to_voc(txt_path, img_path, label)
            if len(objects) == 0:
                continue

            out_img_path = os.path.join(OUTPUT_IMG_DIR, f"{base_name}.jpg")
            out_xml_path = os.path.join(OUTPUT_ANN_DIR, f"{base_name}.xml")
            shutil.copyfile(img_path, out_img_path)
            create_voc_xml(out_img_path, w, h, objects, out_xml_path)
        except Exception as e:
            print(f"[ERROR] {img_path} — {e}")

# Run on both train/test folders
process_folder(TRAIN_DIR)
process_folder(TEST_DIR)


In [ ]:
label_map_path = "/content/fruit_images_pascal/label_map.pbtxt"
with open(label_map_path, "w") as f:
    for idx, name in REVERSE_CLASS_MAP.items():
        f.write("item {\n")
        f.write(f"  id: {idx + 1}\n")
        f.write(f"  name: '{name}'\n")
        f.write("}\n")


In [ ]:
import os
import shutil
import glob

# Fruit classes to keep
FRUIT_CLASSES = [
    'Apple', 'Avocado', 'Banana', 'Blackberry', 'Blueberry', 'Dates', 'Dragonfruit',
    'Fig', 'Grapes', 'Guava', 'Kiwi', 'Lemon', 'Mango', 'Olive', 'Orange', 'Papaya',
    'Pear', 'Pineapple', 'Pomegranate', 'Rambutan', 'Strawberry', 'Tomato', 'Watermelon'
]
FRUIT_CLASSES = [cls.lower() for cls in FRUIT_CLASSES]

# Local source and output paths
SOURCE_ROOT = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Fruit and Vegetables"
DEST_ROOT = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Filtered_Classification"

def copy_images(split):
    input_dir = os.path.join(SOURCE_ROOT, split)
    if not os.path.exists(input_dir):
        print(f"[SKIP] {split} folder not found.")
        return

    for class_dir in os.listdir(input_dir):
        class_path = os.path.join(input_dir, class_dir)
        if not os.path.isdir(class_path):
            continue

        class_name = class_dir.strip().lower()
        if class_name not in FRUIT_CLASSES:
            continue  # Skip non-fruits

        dest_dir = os.path.join(DEST_ROOT, split, class_name)
        os.makedirs(dest_dir, exist_ok=True)

        for img_file in glob.glob(os.path.join(class_path, '*.*')):
            try:
                shutil.copy(img_file, dest_dir)
            except Exception as e:
                print(f"[ERROR] Copying {img_file} — {e}")

# Run for all
for split in ['train', 'validation', 'test']:
    copy_images(split)

print("✅ Fruit-only dataset created at:", DEST_ROOT)


✅ Fruit-only dataset created at: C:\Users\DM77\Documents\Detection Data\All_Datasets\Filtered_Classification


The fruits we are interested in:



In [ ]:
import os
import shutil
import glob

# Source and target paths
SOURCE_ROOT = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Filtered_Classification"
DEST_ROOT = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\All_Fruits"
os.makedirs(DEST_ROOT, exist_ok=True)

# Combine train, validation, and test into one
for split in ['train', 'validation', 'test']:
    split_path = os.path.join(SOURCE_ROOT, split)
    if not os.path.exists(split_path):
        continue

    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        dest_class_path = os.path.join(DEST_ROOT, class_name.lower())
        os.makedirs(dest_class_path, exist_ok=True)

        for img_file in glob.glob(os.path.join(class_path, '*.*')):
            try:
                filename = os.path.basename(img_file)
                # Rename prevent collisions
                new_filename = f"{split}_{filename}"
                shutil.copy(img_file, os.path.join(dest_class_path, new_filename))
            except Exception as e:
                print(f"[ERROR] Copying {img_file} — {e}")

print("✅ Combined dataset created at:", DEST_ROOT)


✅ Combined dataset created at: C:\Users\DM77\Documents\Detection Data\All_Datasets\All_Fruits


In [ ]:
from collections import Counter

base_dir = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\All_Fruits"
counts = Counter()

for class_name in os.listdir(base_dir):
    class_dir = os.path.join(base_dir, class_name)
    if os.path.isdir(class_dir):
        num_images = len(glob.glob(os.path.join(class_dir, '*.*')))
        counts[class_name] = num_images


for fruit, count in counts.most_common():
    print(f"{fruit:<15}: {count}")


orange         : 4044
pomegranate    : 3514
banana         : 2946
grapes         : 2611
pineapple      : 2497
strawberry     : 2353
watermelon     : 2187
kiwi           : 2165
mango          : 2161
lemon          : 2129
apple          : 1844
avocado        : 1784
dragonfruit    : 1585
pear           : 1454
blueberry      : 1245
tomato         : 1189
blackberry     : 1179
rambutan       : 1167
olive          : 1082
guava          : 1074
fig            : 1062
dates          : 549


In [ ]:
import os
import shutil
import glob

# Top 17 fruit classes
TOP_17_CLASSES = [
    'orange', 'pomegranate', 'banana', 'grapes', 'pineapple', 'strawberry',
    'watermelon', 'kiwi', 'mango', 'lemon', 'apple', 'avocado', 'dragonfruit',
    'pear', 'blueberry', 'tomato', 'blackberry'
]

SOURCE_DIR = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\All_Fruits"
DEST_DIR = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Final_17_Classes"

os.makedirs(DEST_DIR, exist_ok=True)

for fruit in os.listdir(SOURCE_DIR):
    fruit_lower = fruit.strip().lower()
    if fruit_lower not in TOP_17_CLASSES:
        continue

    source_fruit_dir = os.path.join(SOURCE_DIR, fruit)
    dest_fruit_dir = os.path.join(DEST_DIR, fruit_lower)
    os.makedirs(dest_fruit_dir, exist_ok=True)

    for img_file in glob.glob(os.path.join(source_fruit_dir, '*.*')):
        try:
            shutil.copy(img_file, dest_fruit_dir)
        except Exception as e:
            print(f"[ERROR] {img_file} — {e}")

print("✅ Final 17-class dataset created at:", DEST_DIR)


✅ Final 17-class dataset created at: C:\Users\DM77\Documents\Detection Data\All_Datasets\Final_17_Classes


In [ ]:
pip install imgaug


  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\DM77\tf-gpu\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
pip install numpy==1.24.3


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\DM77\tf-gpu\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
import os
import glob
import shutil
from PIL import Image
from imgaug import augmenters as iaa
import imgaug as ia
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom

SOURCE_DIR = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Final_17_Classes"
DEST_IMG_DIR = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Augmented_Detection\images"
DEST_XML_DIR = r"C:\Users\DM77\Documents\Detection Data\All_Datasets\Augmented_Detection\annotations"
os.makedirs(DEST_IMG_DIR, exist_ok=True)
os.makedirs(DEST_XML_DIR, exist_ok=True)

# augmentations
AUG = iaa.Sequential([
    iaa.Fliplr(0.5),
    iaa.Multiply((0.8, 1.2)),           # brightness
    iaa.LinearContrast((0.75, 1.25)),  # contrast
    iaa.GaussianBlur(sigma=(0.0, 1.0)),
    iaa.Affine(rotate=(-10, 10), scale=(0.9, 1.1))
])

# generate VOC XML
def create_voc_xml(filename, img_shape, bbox, label, output_path):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'filename').text = filename
    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(img_shape[1])
    ET.SubElement(size, 'height').text = str(img_shape[0])
    ET.SubElement(size, 'depth').text = str(img_shape[2])

    obj = ET.SubElement(annotation, 'object')
    ET.SubElement(obj, 'name').text = label
    bbox_tag = ET.SubElement(obj, 'bndbox')
    ET.SubElement(bbox_tag, 'xmin').text = str(int(bbox[0]))
    ET.SubElement(bbox_tag, 'ymin').text = str(int(bbox[1]))
    ET.SubElement(bbox_tag, 'xmax').text = str(int(bbox[2]))
    ET.SubElement(bbox_tag, 'ymax').text = str(int(bbox[3]))

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent="  ")
    with open(output_path, 'w') as f:
        f.write(xml_str)


for fruit in os.listdir(SOURCE_DIR):
    class_dir = os.path.join(SOURCE_DIR, fruit)
    if not os.path.isdir(class_dir):
        continue

    image_paths = glob.glob(os.path.join(class_dir, "*.jpg"))
    for idx, img_path in enumerate(image_paths):
        try:
            img = Image.open(img_path).convert("RGB")
            img_np = np.array(img)
            h, w = img_np.shape[:2]

            # Original box (whole image)
            bbs = ia.BoundingBoxesOnImage([ia.BoundingBox(x1=0, y1=0, x2=w, y2=h)], shape=img_np.shape)

            # Augment
            for i in range(2):  # generate 2 augmentations per image
                img_aug, bbs_aug = AUG(image=img_np, bounding_boxes=bbs)
                bb = bbs_aug[0]  # get transformed bounding box

                out_name = f"{fruit}_{idx}_{i}.jpg"
                out_img_path = os.path.join(DEST_IMG_DIR, out_name)
                out_xml_path = os.path.join(DEST_XML_DIR, out_name.replace('.jpg', '.xml'))

                Image.fromarray(img_aug).save(out_img_path)
                create_voc_xml(out_name, img_aug.shape, [bb.x1, bb.y1, bb.x2, bb.y2], fruit.lower(), out_xml_path)
        except Exception as e:
            print(f"[ERROR] {img_path} — {e}")
